# FinSight · Company news tool (Finnhub)

**Owner:** P1 (Data)

Answers *"what's been in the news about NVIDIA this week?"* and supplies headlines for the Analysis agent's sentiment work (P2) and for citations in reports.

**Why a separate tool from prices, but a shared client?**
- `news.py` and `prices.py` do different jobs (a list of articles vs one number) and feed different teammates.
- Both use the **same Finnhub key and the same ~60 calls/minute budget**, so they share one client (`finnhub_client.py`) with one rate limiter, just as `edgar.py` and `xbrl.py` share `sec_client.py`.

> Exploration version. The same code lives in `src/finsight/tools/` in the repo; fix bugs there first, then copy here.

## 1. Setup
Uses the same `FINNHUB_API_KEY` as the price tool, from the `.env` in your Drive (Colab) or the repo's `.env` (laptop).

In [ ]:
# Colab already has these; this is for running locally
#%pip install -q requests pandas matplotlib python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ENV_PATH = "/content/drive/MyDrive/Colab Notebooks/6_CMPE-259/.env"
except ImportError:          # running locally (VS Code / Jupyter): use the repo's .env
    ENV_PATH = ".env"

assert os.path.exists(ENV_PATH), f".env not found at {ENV_PATH}"
load_dotenv(ENV_PATH, override=True)

key = os.getenv("FINNHUB_API_KEY", "")
assert key and "your_finnhub_key" not in key, "Add FINNHUB_API_KEY to your .env"
print("FINNHUB_API_KEY loaded:", key[:4] + "…")

## 2. Shared Finnhub client
One place for the key (sent in a header, never in the URL), the rate limit (≥ 1.1 s between calls) and an optional disk cache in `data/cache/finnhub/`.
The last line exposes it as `finnhub_client.get(...)`, exactly as `news.py` calls it in the repo.

In [ ]:
"""Shared HTTP client for Finnhub (used by prices.py and news.py).

Like sec_client.py for the SEC: one place for the key, the rate limit and the cache.
All Finnhub tools share ONE free-tier budget (about 60 calls/minute per key), so
they must share one rate limiter; two separate limiters could exceed it together.

  - Key: FINNHUB_API_KEY in .env (free from https://finnhub.io). Sent in the
    X-Finnhub-Token header, so it never appears in URLs or logs.
  - Rate limit: >= 1.1 s between calls.
  - Optional disk cache (data/cache/finnhub/) for responses that don't change much.
"""


import hashlib
import json
import os
import time
from pathlib import Path

import requests


BASE_URL = "https://finnhub.io/api/v1"
CACHE_DIR = Path(os.getenv("FINSIGHT_CACHE_DIR", "data/cache/sec")).parent / "finnhub"
MIN_INTERVAL = 1.1  # seconds between calls -> under 60 calls/minute

_session = requests.Session()
_last_call = 0.0


def api_key() -> str:
    key = os.getenv("FINNHUB_API_KEY", "")
    if not key:
        raise RuntimeError("Set FINNHUB_API_KEY in your .env (free key from https://finnhub.io)")
    return key


def _cache_path(path: str, params: dict) -> Path:
    key = hashlib.sha256(f"{path}?{sorted(params.items())}".encode()).hexdigest()[:24]
    return CACHE_DIR / f"{key}.json"


def get(path: str, params: dict, ttl: int = 0):
    """GET https://finnhub.io/api/v1{path}. ttl > 0 reuses a saved response for ttl seconds."""
    cache_file = _cache_path(path, params) if ttl > 0 else None
    if cache_file and cache_file.exists() and time.time() - cache_file.stat().st_mtime < ttl:
        return json.loads(cache_file.read_text())

    global _last_call
    for attempt in range(3):
        wait = MIN_INTERVAL - (time.time() - _last_call)
        if wait > 0:
            time.sleep(wait)
        _last_call = time.time()
        resp = _session.get(BASE_URL + path, params=params,
                            headers={"X-Finnhub-Token": api_key()}, timeout=20)
        if resp.status_code == 429 or resp.status_code >= 500:
            time.sleep(2 ** attempt * 2)  # 2s, 4s, 8s
            continue
        if resp.status_code in (401, 403):
            raise RuntimeError("Finnhub rejected the request: check FINNHUB_API_KEY, "
                               "or this endpoint/date range isn't on the free plan")
        resp.raise_for_status()
        data = resp.json()
        if cache_file:
            cache_file.parent.mkdir(parents=True, exist_ok=True)
            cache_file.write_text(json.dumps(data))
        return data
    resp.raise_for_status()
    return resp.json()


from types import SimpleNamespace
finnhub_client = SimpleNamespace(get=get, api_key=api_key)

## 3. The news tool

Finnhub's `/company-news` returns `headline`, `summary`, `source` (publisher), `url`, `datetime` (Unix), `related` and `category`.

**What the tool adds**
- **Newest first**, with `published_at` as an ISO UTC time.
- **Duplicates removed:** the same story syndicated to several sites (same URL, or the same headline ignoring punctuation and case).
- **Caching:** 15 min if the range includes today (new articles keep coming), 7 days for past ranges.
- **Range checks:** the free plan covers about one year back.
- **Relevance filter:** Finnhub tags an article to a ticker even when the company is only mentioned in passing (e.g. a Motley Fool piece about SpaceX tagged NVDA). By default we keep only articles whose headline or summary names the company: `high` = headline, `medium` = summary. Names come from `ALIASES`; add your companies there.
- **`citation()`**: `"Reuters, 2026-10-24: headline (url)"`, ready for the Report agent.

In [ ]:
"""Company news tool (Finnhub free tier).

Answers "what's been in the news about NVIDIA this week?" and supplies headlines
for the Analysis agent's sentiment work (P2) and for citations in reports.

Source: https://finnhub.io/api/v1/company-news?symbol=NVDA&from=2026-10-19&to=2026-10-26
Each article: headline, summary, source (publisher), url, datetime (Unix), related, category, id.

Rules:
  - Same key and same 60 calls/minute budget as prices.py (shared finnhub_client).
  - Free plan: North American companies; roughly the last year of news.
  - Responses are cached on disk: 15 min if the range includes today (new articles
    keep arriving), 7 days for ranges that are fully in the past.
  - Duplicates (same URL, or same headline from syndication) are removed.
  - Relevance: Finnhub tags an article to a ticker even when the company is only
    mentioned in passing (e.g. a Motley Fool piece about SpaceX that lists Nvidia
    in a disclosure). We keep an article only if its headline or summary names
    the company: "high" = in the headline, "medium" = in the summary only.
  - Articles are evidence, not advice: agents cite them with source + url + date.
"""


import re
from dataclasses import asdict, dataclass, field
from datetime import date, datetime, timedelta, timezone


RECENT_TTL = 15 * 60            # range includes today: refetch after 15 minutes
PAST_TTL = 7 * 24 * 60 * 60     # range fully in the past: articles won't change
MAX_DAYS_BACK = 365             # free plan covers about one year

# Names to look for in headlines/summaries (the ticker itself is always included).
# Add your companies here if they're missing; or pass names=[...] to get_company_news.
ALIASES: dict[str, list[str]] = {
    "NVDA": ["Nvidia"],
    "AAPL": ["Apple", "iPhone"],
    "MSFT": ["Microsoft"],
    "AMZN": ["Amazon", "AWS"],
    "GOOGL": ["Alphabet", "Google"],
    "GOOG": ["Alphabet", "Google"],
    "META": ["Meta Platforms", "Meta", "Facebook", "Instagram"],
    "TSLA": ["Tesla"],
    "JPM": ["JPMorgan", "JP Morgan", "J.P. Morgan"],
    "AMD": ["Advanced Micro Devices"],
    "INTC": ["Intel"],
}
RELEVANCE_RANK = {"low": 0, "medium": 1, "high": 2}


@dataclass
class NewsArticle:
    ticker: str
    headline: str
    summary: str
    source: str               # publisher, e.g. "Reuters"
    url: str
    published_at: str         # ISO UTC
    category: str = ""
    related: list[str] = field(default_factory=list)
    article_id: int | None = None
    relevance: str = ""       # "high" (named in headline), "medium" (summary), "low" (neither)
    provider: str = "Finnhub"

    @property
    def date(self) -> str:
        return self.published_at[:10]

    def to_dict(self) -> dict:
        return asdict(self)

    def citation(self) -> str:
        """Short reference for reports, e.g. 'Reuters, 2026-10-24: Nvidia ... (url)'."""
        return f"{self.source}, {self.date}: {self.headline} ({self.url})"


def _norm(headline: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", headline.lower()).strip()


def company_pattern(ticker: str, names: list[str] | None = None) -> re.Pattern:
    """Regex matching the ticker or any company name as a whole word ("Meta" but not "metaverse")."""
    terms = [ticker.upper(), *(names if names is not None else ALIASES.get(ticker.upper(), []))]
    alts = "|".join(re.escape(t) for t in sorted(set(terms), key=len, reverse=True))
    return re.compile(rf"(?<![A-Za-z0-9])\$?(?:{alts})(?![A-Za-z0-9])", re.IGNORECASE)


def relevance(headline: str, summary: str, pattern: re.Pattern) -> str:
    if pattern.search(headline):
        return "high"
    if pattern.search(summary):
        return "medium"
    return "low"


def parse_articles(ticker: str, rows: list[dict], names: list[str] | None = None,
                   min_relevance: str = "low") -> list[NewsArticle]:
    """Finnhub rows -> NewsArticle list, newest first, duplicates removed. Pure function.

    min_relevance="medium" drops articles that never name the company.
    """
    pattern = company_pattern(ticker, names)
    floor = RELEVANCE_RANK[min_relevance]
    out: list[NewsArticle] = []
    seen_urls: set[str] = set()
    seen_heads: set[str] = set()
    for r in sorted(rows or [], key=lambda r: r.get("datetime") or 0, reverse=True):
        headline = (r.get("headline") or "").strip()
        url = (r.get("url") or "").strip()
        if not headline or not url or not r.get("datetime"):
            continue
        key = _norm(headline)
        if url in seen_urls or key in seen_heads:
            continue
        summary = (r.get("summary") or "").strip()
        rel = relevance(headline, summary, pattern)
        if RELEVANCE_RANK[rel] < floor:
            continue
        seen_urls.add(url)
        seen_heads.add(key)
        out.append(NewsArticle(
            ticker=ticker.upper(),
            headline=headline,
            summary=summary,
            source=(r.get("source") or "").strip(),
            url=url,
            published_at=datetime.fromtimestamp(r["datetime"], tz=timezone.utc)
                                 .isoformat(timespec="seconds"),
            category=r.get("category") or "",
            related=[x.strip() for x in (r.get("related") or "").split(",") if x.strip()],
            article_id=r.get("id"),
            relevance=rel,
        ))
    return out


def get_company_news(ticker: str, days: int = 7, start: str | None = None,
                     end: str | None = None, limit: int | None = None,
                     min_relevance: str = "medium", names: list[str] | None = None,
                     today: date | None = None) -> list[NewsArticle]:
    """News about one company, newest first.

    get_company_news("NVDA")                       -> last 7 days, articles that name Nvidia
    get_company_news("NVDA", days=30, limit=20)    -> last 30 days, 20 newest
    get_company_news("NVDA", start="2026-08-20", end="2026-08-31")  -> around an earnings date
    min_relevance: "high" = company in the headline; "medium" (default) = headline or
                   summary; "low" = everything Finnhub tagged (incl. passing mentions).
    """
    if min_relevance not in RELEVANCE_RANK:
        raise ValueError(f"min_relevance must be one of {list(RELEVANCE_RANK)}")
    today = today or date.today()
    end_d = date.fromisoformat(end) if end else today
    start_d = date.fromisoformat(start) if start else end_d - timedelta(days=days)
    if start_d > end_d:
        raise ValueError(f"start {start_d} is after end {end_d}")
    if (today - start_d).days > MAX_DAYS_BACK:
        raise ValueError(f"Free plan covers about {MAX_DAYS_BACK} days back; {start_d} is too old")

    ttl = RECENT_TTL if end_d >= today else PAST_TTL
    rows = finnhub_client.get("/company-news",
                              {"symbol": ticker.upper(), "from": start_d.isoformat(),
                               "to": end_d.isoformat()}, ttl=ttl)
    if isinstance(rows, dict):  # Finnhub reports some problems as {"error": "..."}
        raise ValueError(f"Finnhub: {rows.get('error', rows)}")
    articles = parse_articles(ticker, rows, names=names, min_relevance=min_relevance)
    return articles[:limit] if limit else articles  # limit AFTER filtering


def get_news_for(tickers: list[str], days: int = 7, limit: int | None = 10,
                 min_relevance: str = "medium") -> dict[str, list | str]:
    """News for several companies. A failed ticker gets an error message instead of a list."""
    out: dict[str, list | str] = {}
    for t in tickers:
        try:
            out[t.upper()] = get_company_news(t, days=days, limit=limit, min_relevance=min_relevance)
        except Exception as e:  # noqa: BLE001 — one bad ticker shouldn't stop the rest
            out[t.upper()] = f"{type(e).__name__}: {e}"
    return out

## 4. Check the logic offline
Made-up articles, no internet. Two are duplicates and one has no headline, so only 2 of the 5 should survive.

In [ ]:
ROWS = [
    {"datetime": 1_761_400_000, "headline": "Nvidia unveils new chip", "id": 1, "related": "NVDA",
     "source": "Reuters", "summary": "Details...", "url": "https://example.com/a", "category": "company"},
    {"datetime": 1_761_300_000, "headline": "NVIDIA unveils new chip!", "id": 2, "related": "NVDA",
     "source": "Yahoo", "summary": "Syndicated copy", "url": "https://example.com/b", "category": "company"},
    {"datetime": 1_761_200_000, "headline": "Nvidia supplier update", "id": 3, "related": "NVDA,TSM",
     "source": "CNBC", "summary": "", "url": "https://example.com/a", "category": "company"},
    {"datetime": 1_761_500_000, "headline": "Nvidia earnings preview", "id": 4, "related": "NVDA",
     "source": "MarketWatch", "summary": "Preview", "url": "https://example.com/c", "category": "company"},
    {"datetime": 1_761_100_000, "headline": "", "id": 5, "related": "NVDA",
     "source": "X", "summary": "", "url": "https://example.com/d", "category": "company"},
]

arts = parse_articles("nvda", ROWS)
for a in arts:
    print(a.citation())

assert [a.article_id for a in arts] == [4, 1]
print("\n✓ newest first; syndicated duplicate, repeated URL and empty headline removed")

## 5. Try it on live data

In [ ]:
import pandas as pd

# Ask the user to enter a company ticker.
TICKER = input("Enter a stock ticker (e.g., NVDA, AAPL, MSFT): ").strip().upper()

articles = get_company_news(TICKER, days=7)
print(f"{len(articles)} articles about {TICKER} in the last 7 days\n")

news_df = pd.DataFrame([a.to_dict() for a in articles])
news_df[["published_at", "relevance", "source", "headline", "url"]].head(15) if not news_df.empty else news_df

In [ ]:
# What the relevance filter removed: everything Finnhub tagged vs articles that name the company
everything = get_company_news(TICKER, days=7, min_relevance="low")
kept_urls = {a.url for a in articles}
dropped = [a for a in everything if a.url not in kept_urls]
print(f"Finnhub tagged {len(everything)} articles; {len(articles)} name {TICKER}; {len(dropped)} dropped, e.g.:")
for a in dropped[:5]:
    print("  -", a.headline[:100])

In [ ]:
# Ready-made citations for the Report agent
for a in articles[:5]:
    print("-", a.citation())

In [ ]:
# News around a specific window, e.g. an earnings date (change the dates)
earnings_window = get_company_news(TICKER, start="2026-08-20", end="2026-09-03")
print(f"{len(earnings_window)} articles between 2026-08-20 and 2026-09-03")
pd.DataFrame([a.to_dict() for a in earnings_window[:10]],
             columns=["published_at", "source", "headline", "url"])

In [ ]:
# Running the same request again comes from the disk cache (no API call)
import time
t0 = time.time()
again = get_company_news(TICKER, days=7)
print(f"second call took {time.time() - t0:.3f} s, same number of articles: {len(again) == len(articles)}")

## 6. All 10 companies
About one call per second. Counts show how much coverage each company gets; this matters for P2's sentiment work, since a company with few articles gives a small sample (US-12: show the sample size).

In [ ]:
COMPANIES = ["NVDA", "AAPL", "MSFT", "AMZN", "GOOGL", "META", "TSLA", "JPM", "AMD", "INTC"]  # ← your 10

all_news = get_news_for(COMPANIES, days=7, limit=None)
rows = []
for t, arts in all_news.items():
    if isinstance(arts, str):
        rows.append({"ticker": t, "articles": None, "error": arts})
    else:
        rows.append({"ticker": t, "articles": len(arts),
                     "publishers": len({a.source for a in arts}),
                     "latest": arts[0].published_at if arts else None,
                     "latest_headline": arts[0].headline[:80] if arts else None})
coverage = pd.DataFrame(rows)
coverage

In [ ]:
import matplotlib.pyplot as plt

ok = coverage.dropna(subset=["articles"]).sort_values("articles")
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(ok["ticker"], ok["articles"], color="#2b6cb0")
ax.set_xlabel("Articles in the last 7 days (after removing duplicates)")
ax.set_title("News coverage per company (Finnhub)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## Notes and next steps
- **Evidence, not advice:** agents cite articles (source, date, link) and never turn headlines into buy / sell advice.
- **For P2 (sentiment):** headlines and summaries are the input sentences; hand over `to_dict()` rows. Consider saving them in a `news_articles` table in our SQLite database so the labelled set stays fixed.
- **Second provider (optional):** Alpha Vantage `NEWS_SENTIMENT` adds sentiment scores but allows only 25 calls/day: cache everything. It would be a second function in `news.py` returning the same `NewsArticle` shape, with `provider="AlphaVantage"`.
- **Free plan limits:** North American companies, about one year back.